# AI Detector — Google Colab Training
**UnifiedFusionNet v1** | 16 branches | 19.2M params

---
**Before running:**
1. `Runtime → Change runtime type → GPU` (T4 free / A100 Colab Pro)
2. Run all cells top to bottom
3. Checkpoints auto-save to your Google Drive every epoch

## Step 1 — Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected! Go to Runtime → Change runtime type → GPU")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"CUDA : {torch.version.cuda}")

# Auto-select batch size based on VRAM — bigger batch = better GPU utilization
if vram_gb >= 40:          # A100 80GB / A100 40GB
    BATCH_SIZE = 32;  IMG_SIZE = 640;  GRAD_ACCUM = 2
elif vram_gb >= 24:        # A100 24GB / 3090
    BATCH_SIZE = 16;  IMG_SIZE = 640;  GRAD_ACCUM = 2
elif vram_gb >= 16:        # V100 16GB / L4 24GB / T4 16GB
    BATCH_SIZE = 8;   IMG_SIZE = 512;  GRAD_ACCUM = 4
elif vram_gb >= 10:        # T4 10GB edge case
    BATCH_SIZE = 4;   IMG_SIZE = 512;  GRAD_ACCUM = 8
else:
    BATCH_SIZE = 2;   IMG_SIZE = 384;  GRAD_ACCUM = 16

print(f"\nAuto config → batch={BATCH_SIZE}, img={IMG_SIZE}px, accum={GRAD_ACCUM}")
print(f"Effective batch size = {BATCH_SIZE * GRAD_ACCUM}")

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All checkpoints and models save here — persists after Colab disconnects
DRIVE_DIR = '/content/drive/MyDrive/ai_detector_models'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive dir: {DRIVE_DIR}")
print("Files in drive dir:", os.listdir(DRIVE_DIR) if os.listdir(DRIVE_DIR) else '(empty — first run)')

## Step 3 — Clone Project from GitHub

In [ ]:
import os, sys

REPO_URL  = 'https://github.com/wwczwbcv6n-cpu/ai_detector_core.git'
REPO_DIR  = '/content/ai_detector_core'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest changes...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    print('Cloning repo...')
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print(f"Working dir: {os.getcwd()}")

## Step 4 — Install Dependencies

In [ ]:
# Colab already has: torch, torchvision, numpy, Pillow, opencv, scikit-learn, tqdm
# Only install what's missing
!pip install -q pillow-heif rawpy imageio PyWavelets scikit-image
!pip install -q timm      # required by EfficientNet in model_prnu.py
!pip install -q yt-dlp    # required for YouTube/video URL training
print('Dependencies installed.')

## Step 5 — Setup Paths & Matplotlib Backend

In [ ]:
import matplotlib
matplotlib.use('Agg')   # non-interactive — prevents crashes on Colab
import matplotlib.pyplot as plt
import os, sys, json, gc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
_pil_bilinear = getattr(Image, 'Resampling', Image).BILINEAR  # Pillow 10+ compat
from IPython.display import clear_output, display

REPO_DIR   = '/content/ai_detector_core'
DRIVE_DIR  = '/content/drive/MyDrive/ai_detector_models'
MODELS_DIR = os.path.join(REPO_DIR, 'models')
DATA_DIR   = os.path.join(REPO_DIR, 'data')

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATA_DIR,   exist_ok=True)

# Restore checkpoint from Drive if it exists (resume after disconnect)
UNIFIED_CKPT  = os.path.join(MODELS_DIR, 'ai_detector_unified_v1.pth')
DRIVE_CKPT    = os.path.join(DRIVE_DIR,  'ai_detector_unified_v1.pth')
if not os.path.exists(UNIFIED_CKPT) and os.path.exists(DRIVE_CKPT):
    import shutil
    shutil.copy(DRIVE_CKPT, UNIFIED_CKPT)
    print(f'Restored checkpoint from Drive: {DRIVE_CKPT}')

device = torch.device('cuda')
print(f'Device: {device} — {torch.cuda.get_device_name(0)}')

## Step 6 — Load Model

In [ ]:
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
from model_prnu import UnifiedFusionNet, check_checkpoint_compat

model = UnifiedFusionNet(
    prnu_in_features=64,
    gradient_checkpointing=True,   # saves ~300-600 MB VRAM
).to(device)

# Load existing checkpoint if available (resume training)
if os.path.exists(UNIFIED_CKPT):
    ckpt = torch.load(UNIFIED_CKPT, map_location=device, weights_only=True)
    check_checkpoint_compat(model, ckpt.get('model_state_dict', ckpt))
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    print(f'Loaded checkpoint: {UNIFIED_CKPT}')
else:
    print('No checkpoint found — starting fresh')

model.param_summary()
print(f'\nVRAM after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## Step 7 — Dataset & Training Config

In [ ]:
from prnu_features import extract_prnu_features_fullres, extract_prnu_map
from format_analyzer import get_shared_analyzer

# ─── Colab-optimised settings ─────────────────────────────────────────────
EPOCHS           = 10
LR               = 1e-3
SAVE_EVERY       = 1        # save checkpoint every N epochs
FORMAT_FEATURE_DIM = 128    # FormatForensicsBranch input dim
AUX_WEIGHTS = dict(    # per-branch aux loss weights
    prnu=0.35, halluc=0.15, prnu_spatial=0.20,
    gan=0.25,  cmos=0.20,   color_incon=0.20,
    flow=0.15, motion=0.15, format=0.20,
)

class ColabImageDataset(Dataset):
    """
    Simple dataset that scans data/real and data/ai.
    Returns (img_tensor, prnu_feats_64, prnu_map_3x128x128, fmt_feats_128, label).
    NOTE: num_workers MUST be 0 — FormatAnalyzer contains non-picklable state.
    """
    EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

    def __init__(self, paths, labels, img_size=512, augment=True):
        self.paths  = paths
        self.labels = labels
        # Format analyzer — enable_restoration=False for training speed
        self._fmt_analyzer = get_shared_analyzer(enable_restoration=False)
        if augment:
            self.tf = transforms.Compose([
                transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
                transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
            ])
        else:
            self.tf = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
            ])

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        path  = self.paths[idx]
        label = self.labels[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))

        # PRNU scalar features (64-dim)
        try:
            arr  = np.array(img.resize((512, 512), _pil_bilinear), dtype=np.float64) / 255.0
            pf   = extract_prnu_features_fullres(arr, tile_size=128)
            if not np.isfinite(pf).all(): pf = np.zeros(64, dtype=np.float32)
        except Exception:
            pf = np.zeros(64, dtype=np.float32)

        # PRNU spatial map (3×128×128)
        try:
            pm = extract_prnu_map(img, output_size=128)          # (128,128,3)
            if not np.isfinite(pm).all(): pm = np.zeros((128,128,3), dtype=np.float32)
        except Exception:
            pm = np.zeros((128,128,3), dtype=np.float32)

        # Format forensics features (128-dim)
        try:
            ff = self._fmt_analyzer.analyze(path)
            fmt_feats = ff.feature_vector
            if not np.isfinite(fmt_feats).all():
                fmt_feats = np.zeros(FORMAT_FEATURE_DIM, dtype=np.float32)
        except Exception:
            fmt_feats = np.zeros(FORMAT_FEATURE_DIM, dtype=np.float32)

        return (
            self.tf(img),
            torch.from_numpy(pf).float(),
            torch.from_numpy(pm.transpose(2,0,1)).float(),
            torch.from_numpy(fmt_feats).float(),
            torch.tensor(label, dtype=torch.float32),
        )


def scan_data(real_dirs, ai_dirs):
    exts = ColabImageDataset.EXTS
    paths, labels = [], []
    for d in real_dirs:
        if not os.path.isdir(d): continue
        for r, _, fs in os.walk(d):
            for f in fs:
                if os.path.splitext(f)[1].lower() in exts:
                    paths.append(os.path.join(r, f)); labels.append(0)
    for d in ai_dirs:
        if not os.path.isdir(d): continue
        for r, _, fs in os.walk(d):
            for f in fs:
                if os.path.splitext(f)[1].lower() in exts:
                    paths.append(os.path.join(r, f)); labels.append(1)
    return paths, labels

print('Dataset class ready.')

## Step 7b — Upload Your Data (choose one option)

**Option A — Upload from local machine:**
```python
from google.colab import files
uploaded = files.upload()   # select your zip
!unzip -q my_data.zip -d /content/ai_detector_core/data/
```

**Option B — Copy from Google Drive:**
```python
!cp -r /content/drive/MyDrive/my_training_data /content/ai_detector_core/data/
```

**Option C — Use Open Images streaming (no upload needed):**
Just run `train_streaming.py` in Step 9B — it downloads images on-the-fly.

**Expected folder structure:**
```
data/
  real/        ← real camera photos (label=0)
  ai/          ← AI-generated images (label=1)
```

## Step 8 — Training Loop with Auto-Save to Drive

In [ ]:
from sklearn.model_selection import train_test_split
import shutil, time

# ─── Scan local data ──────────────────────────────────────────────────────
paths, labels = scan_data(
    real_dirs=[f'{DATA_DIR}/real', f'{DATA_DIR}/train/REAL'],
    ai_dirs  =[f'{DATA_DIR}/ai',   f'{DATA_DIR}/train/FAKE',
                f'{DATA_DIR}/generated_ai/images'],
)
print(f'Found {labels.count(0) if isinstance(labels, list) else sum(l==0 for l in labels)} real, '
      f'{labels.count(1) if isinstance(labels, list) else sum(l==1 for l in labels)} AI images')

if len(paths) == 0:
    print('\nNo local data found.')
    print('Upload data first (see Step 7b) or use streaming trainer (Step 9B).')
else:
    X_tr, X_val, y_tr, y_val = train_test_split(
        paths, labels, test_size=0.15, random_state=42, stratify=labels
    )

    train_ds = ColabImageDataset(X_tr,  y_tr,  img_size=IMG_SIZE, augment=True)
    val_ds   = ColabImageDataset(X_val, y_val, img_size=IMG_SIZE, augment=False)

    # num_workers=0: FormatAnalyzer (ForensicRecoverer) is not picklable across workers
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

    # ─── Optimizer & Scheduler ────────────────────────────────────────────
    criterion = nn.BCEWithLogitsLoss()
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable, lr=LR, weight_decay=1e-4)
    steps_pe  = max(len(train_dl) // GRAD_ACCUM, 1)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR,
        steps_per_epoch=steps_pe,
        epochs=EPOCHS, pct_start=0.3,
    )
    scaler = torch.amp.GradScaler(device.type, enabled=True)

    # ─── History for live plot ────────────────────────────────────────────
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    def _plot(epoch):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
        ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)
        ax2.plot(history['train_acc'], label='Train'); ax2.plot(history['val_acc'], label='Val')
        ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(True)
        plt.suptitle(f'Epoch {epoch}/{EPOCHS}')
        plt.tight_layout()
        fig.savefig(f'{DRIVE_DIR}/training_plot.png', dpi=100)
        plt.close(fig)
        display(Image.open(f'{DRIVE_DIR}/training_plot.png'))

    def _aux_loss(out, targets):
        if not isinstance(out, tuple): return torch.tensor(0.0, device=device)
        keys = list(AUX_WEIGHTS.keys())
        total = torch.tensor(0.0, device=device)
        for i, aux in enumerate(out[1:]):
            w = AUX_WEIGHTS[keys[i]] if i < len(keys) else 0.15
            total = total + w * criterion(aux, targets)
        return total

    print(f'\nTraining {len(train_ds)} samples | Validating {len(val_ds)} samples')
    print(f'Batch={BATCH_SIZE} | AccumSteps={GRAD_ACCUM} | EffBatch={BATCH_SIZE*GRAD_ACCUM} | Epochs={EPOCHS}\n')

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()

        # ── TRAIN ──────────────────────────────────────────────────────────
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        optimizer.zero_grad()

        for i, (imgs, pf, pm, ff, tgts) in enumerate(train_dl):
            imgs = imgs.to(device); pf = pf.to(device)
            pm   = pm.to(device);   ff = ff.to(device)
            tgts = tgts.to(device).view(-1,1)

            with torch.amp.autocast(device.type, enabled=True):
                out    = model(imgs, pf, pm, format_feats=ff)
                logits = out[0] if isinstance(out, tuple) else out
                loss   = (criterion(logits, tgts) + _aux_loss(out, tgts)) / GRAD_ACCUM

            if torch.isnan(loss):
                optimizer.zero_grad(); continue

            scaler.scale(loss).backward()

            if (i + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(); scheduler.step()

            run_loss += loss.item() * GRAD_ACCUM * imgs.size(0)
            preds    = (torch.sigmoid(logits.detach()) > 0.5).float()
            total   += tgts.size(0)
            correct += (preds == tgts).sum().item()

            if (i + 1) % 20 == 0:
                print(f'  E{epoch} [{i+1}/{len(train_dl)}] '
                      f'loss={run_loss/max(total,1):.4f} '
                      f'acc={correct/max(total,1):.3f} '
                      f'vram={torch.cuda.memory_allocated()/1e9:.1f}GB', flush=True)

            del imgs, pf, pm, ff, tgts, out, logits, loss

        # Flush remaining grads
        if len(train_dl) % GRAD_ACCUM != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()

        train_loss = run_loss / max(total, 1)
        train_acc  = correct  / max(total, 1)

        # ── VALIDATE ───────────────────────────────────────────────────────
        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, pf, pm, ff, tgts in val_dl:
                imgs = imgs.to(device); pf = pf.to(device)
                pm   = pm.to(device);   ff = ff.to(device)
                tgts = tgts.to(device).view(-1,1)
                with torch.amp.autocast(device.type, enabled=True):
                    out    = model(imgs, pf, pm, format_feats=ff)
                    logits = out[0] if isinstance(out, tuple) else out
                    v_loss += criterion(logits, tgts).item() * imgs.size(0)
                preds      = (torch.sigmoid(logits) > 0.5).float()
                v_total   += tgts.size(0)
                v_correct += (preds == tgts).sum().item()

        val_loss = v_loss / max(v_total, 1)
        val_acc  = v_correct / max(v_total, 1)
        elapsed  = time.time() - t0

        print(f'\nEpoch {epoch}/{EPOCHS} ({elapsed:.0f}s) | '
              f'train loss={train_loss:.4f} acc={train_acc:.3f} | '
              f'val loss={val_loss:.4f} acc={val_acc:.3f} | '
              f'lr={optimizer.param_groups[0]["lr"]:.2e}\n')

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        # ── SAVE CHECKPOINT ────────────────────────────────────────────────
        if epoch % SAVE_EVERY == 0:
            local_path = os.path.join(MODELS_DIR, 'ai_detector_unified_v1.pth')
            torch.save({'model_state_dict': model.state_dict(),
                        'epoch': epoch,
                        'val_loss': val_loss, 'val_acc': val_acc}, local_path)
            shutil.copy(local_path, os.path.join(DRIVE_DIR, 'ai_detector_unified_v1.pth'))
            print(f'  Saved checkpoint → Drive ({val_loss:.4f} val loss)')

        clear_output(wait=True)
        _plot(epoch)

        gc.collect()
        torch.cuda.empty_cache()

    print('\nTraining complete!')

## Step 9B — Alternative: Streaming Trainer (No Data Upload Needed)
Downloads real images from Open Images v7 on-the-fly, deletes after each batch.
This is the best option if you don't have your own data ready.

### URL Tips

**Good real-content sources (label=0 — genuine camera footage):**
- Nature/travel YouTube channels (real footage, not AI)
- Stock footage sites with direct MP4 links
- Wikimedia Commons image URLs
- Flickr original photo URLs

**Good AI-content sources (label=1 — AI-generated):**
- YouTube channels showcasing Sora, Runway, Pika Labs
- Midjourney/DALL-E showcase videos
- Any channel that explicitly posts AI-generated art/video

**Playlist example** (all videos in a playlist as real content):
```
https://www.youtube.com/playlist?list=PLxxxxxxxxxx
```
yt-dlp will automatically expand the playlist — each video gives up to 32 frames.

**Mix image + video URLs in the same file:**
```
# Images (downloaded directly)
https://live.staticflickr.com/xxxx/photo.jpg
https://upload.wikimedia.org/wikipedia/commons/photo.png

# Videos (yt-dlp extracts frames)
https://www.youtube.com/watch?v=xxxxx
https://vimeo.com/xxxxx
```

In [ ]:
# Run streaming trainer with your YouTube/URL sources
# --url_file     = real content URLs (label=0)
# --ai_url_file  = AI content URLs  (label=1)
# --total_batches controls how long to run (each batch ~30-90s on T4)

!cd /content/ai_detector_core && python src/train_streaming.py \
    --url_file      data/real_urls.txt \
    --ai_url_file   data/ai_urls.txt \
    --total_batches 200 \
    --batch_real 16 \
    --batch_ai   16 \
    2>&1 | tee /content/drive/MyDrive/ai_detector_models/youtube_training_log.txt

In [ ]:
import os

REPO_DIR  = '/content/ai_detector_core'
DRIVE_DIR = '/content/drive/MyDrive/ai_detector_models'

os.makedirs(os.path.join(REPO_DIR, 'data'), exist_ok=True)

# ── EDIT THESE URLs ────────────────────────────────────────────────────────
# Add your real-content URLs below (one per line, # = comment)
# Supports: YouTube, Vimeo, direct image/video URLs
REAL_URLS = [
    # 'https://www.youtube.com/watch?v=YOUR_REAL_VIDEO_ID',
    # 'https://www.youtube.com/watch?v=Vugj2U_eK24',   # example
]

# Add your AI-generated content URLs below
AI_URLS = [
    # 'https://www.youtube.com/watch?v=YOUR_AI_VIDEO_ID',
]
# ──────────────────────────────────────────────────────────────────────────

real_url_path = os.path.join(REPO_DIR, 'data', 'real_urls.txt')
ai_url_path   = os.path.join(REPO_DIR, 'data', 'ai_urls.txt')

with open(real_url_path, 'w') as f:
    f.write('\n'.join(REAL_URLS) + '\n')

with open(ai_url_path, 'w') as f:
    f.write('\n'.join(AI_URLS) + '\n')

print(f"real_urls.txt → {len(REAL_URLS)} URL(s)")
print(f"ai_urls.txt   → {len(AI_URLS)} URL(s)")

if not REAL_URLS:
    print("\n[!] No real URLs set. Use Open Images streaming instead (Step 9B):")
    print("    python src/train_streaming.py --source val --total_batches 500")
else:
    print("\nURL files ready. Run the training cell below.")

## Step 9A — Train from YouTube / Any Video or Image URL

You can pass **any URL** as training data:
- YouTube videos, Shorts, playlists, channels
- Vimeo, TikTok, Instagram Reels, Twitter/X videos
- Direct image URLs (Flickr, Imgur, Wikimedia…)
- Direct MP4/WebM links

**How it works:**
1. Create a text file with one URL per line
2. Pass it to `train_streaming.py` via `--url_file` (real content) or `--ai_url_file` (AI content)
3. Video URLs → `yt-dlp` downloads → extracts 32 frames per video → trains → deletes
4. Image URLs → direct download → trains → deletes

In [ ]:
# Streaming trainer — UnifiedFusionNet v1
# Starts fresh every time (--no_resume), uses your URL files

!cd /content/ai_detector_core && python src/train_streaming.py \
    --url_file      data/real_urls.txt \
    --ai_url_file   data/ai_urls.txt \
    --total_batches 1000 \
    --batch_real    32 \
    --batch_ai      32 \
    --no_resume \
    2>&1 | tee /content/drive/MyDrive/ai_detector_models/streaming_log.txt

## Step 10 — Download Trained Model

In [ ]:
from google.colab import files
import os

model_path = '/content/ai_detector_core/models/ai_detector_unified_v1.pth'
if os.path.exists(model_path):
    size_mb = os.path.getsize(model_path) / 1e6
    print(f'Downloading model ({size_mb:.0f} MB)...')
    files.download(model_path)
else:
    print('Model not found — run training first.')

## Troubleshooting

| Error | Fix |
|---|---|
| `CUDA out of memory` | Reduce `BATCH_SIZE` to 2, or `IMG_SIZE` to 384 |
| `ModuleNotFoundError: timm` | Run `!pip install timm` |
| `NaN loss` | Reduce `LR` to `1e-4` |
| Runtime disconnected | Re-run from Step 5 — checkpoint auto-restores from Drive |
| Training very slow | Make sure GPU is selected: Runtime → Change runtime type → GPU |
| `No data found` | Upload data (Step 7b) or use streaming trainer (Step 9B) |
| `FileNotFoundError: real_urls.txt` | Run the URL setup cell in Step 9A first |
| `No valid URLs found` | Remove placeholder URLs and add real ones in the REAL_URLS list |
| `yt-dlp not installed` | Run `!pip install yt-dlp` (Step 4 now does this automatically) |
| `Only 0 images. Skipping.` | All URLs failed — check URLs are valid and not placeholder text |